## Developer engagement features (activity + contact)

Join: `activity_clean.dev_contact` → `contact_clean.developer_id`.

| Theme | What we measure |
|--------|------------------|
| **Recency** | How recently the developer engaged (relative to dataset max date), windowed event counts, DevZone login gap |
| **Intensity** | Volume, events per active day / month, capped score concentration |
| **Duration** | Span from first to last activity, distinct active days vs calendar span |
| **Quality** | Capped scores, attendance where recorded, diversity of activity types |

Output: one row per `developer_id` with contact attributes plus aggregates (developers with no activity get zeros/NULLs where appropriate).

In [ ]:
import duckdb
import pandas as pd

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

### Feature SQL

- **Reference time** `ref_ts`: latest `activity_date` in `activity_clean` so recency is comparable in a static extract.
- **Score handling**: `LEAST(COALESCE(activity_score, 0), 100)` per project log (outliers capped).
- **Spine**: `contact_clean` left join to per-developer activity aggregates so CRM dates (`first_activity_date`, `last_activity_date`, `devzone_last_login_date`) sit beside behavioral aggregates.

In [ ]:
ENGAGEMENT_FEATURES_SQL = """
CREATE OR REPLACE TABLE developer_engagement_features AS
WITH params AS (
    SELECT COALESCE(MAX(activity_date), CURRENT_TIMESTAMP) AS ref_ts
    FROM activity_clean
    WHERE activity_date IS NOT NULL
),
base AS (
    SELECT
        a.dev_contact AS developer_id,
        a.activity_date,
        a.activity_id,
        LEAST(COALESCE(a.activity_score, 0), 100) AS score_capped,
        a.activity_type,
        a.activity_role,
        a.activity_name,
        NULLIF(TRIM(a.activity_attendance), '') IS NOT NULL AS has_attendance,
        LOWER(TRIM(COALESCE(a.activity_attendance, ''))) IN ('yes', 'attended', 'present', 'true', '1') AS attended_positive
    FROM activity_clean a
    WHERE a.dev_contact IS NOT NULL
),
act AS (
    SELECT
        b.developer_id,
        COUNT(*) AS n_activity_rows,
        COUNT(DISTINCT b.activity_id) AS n_distinct_activity_id,
        MIN(b.activity_date) AS first_activity_ts,
        MAX(b.activity_date) AS last_activity_ts,
        COUNT(DISTINCT CAST(b.activity_date AS DATE)) AS n_active_days,
        AVG(b.score_capped) AS mean_score_capped,
        STDDEV_POP(b.score_capped) AS std_score_capped,
        MAX(b.score_capped) AS max_score_capped,
        quantile_cont(b.score_capped, 0.5) AS median_score_capped,
        COUNT(DISTINCT b.activity_type) AS n_distinct_activity_types,
        COUNT(DISTINCT b.activity_role) AS n_distinct_activity_roles,
        SUM(CASE WHEN b.score_capped >= 50 THEN 1 ELSE 0 END) AS n_high_score_events,
        SUM(CASE WHEN b.has_attendance THEN 1 ELSE 0 END) AS n_attendance_known,
        SUM(CASE WHEN b.attended_positive THEN 1 ELSE 0 END) AS n_attended_positive,
    FROM base b
    WHERE b.activity_date IS NOT NULL
    GROUP BY b.developer_id
),
win AS (
    SELECT
        b.developer_id,
        COUNT(*) FILTER (
            WHERE b.activity_date >= (SELECT ref_ts FROM params) - INTERVAL 30 DAY
        ) AS n_events_last_30d,
        COUNT(*) FILTER (
            WHERE b.activity_date >= (SELECT ref_ts FROM params) - INTERVAL 90 DAY
        ) AS n_events_last_90d,
        COUNT(*) FILTER (
            WHERE b.activity_date >= (SELECT ref_ts FROM params) - INTERVAL 180 DAY
        ) AS n_events_last_180d,
        COUNT(*) FILTER (
            WHERE b.activity_date >= (SELECT ref_ts FROM params) - INTERVAL 365 DAY
        ) AS n_events_last_365d,
        SUM(b.score_capped) FILTER (
            WHERE b.activity_date >= (SELECT ref_ts FROM params) - INTERVAL 90 DAY
        ) AS sum_score_last_90d
    FROM base b
    WHERE b.activity_date IS NOT NULL
    GROUP BY b.developer_id
),
act_full AS (
    SELECT
        a.*,
        w.n_events_last_30d,
        w.n_events_last_90d,
        w.n_events_last_180d,
        w.n_events_last_365d,
        w.sum_score_last_90d
    FROM act a
    LEFT JOIN win w ON w.developer_id = a.developer_id
)
SELECT
    c.developer_id,
    c.country,
    c.region,
    c.zone,
    c.territory,
    c.development_areas,
    c.industry_segment_vertical,
    c.first_program_application_date,
    c.first_activity_date AS contact_first_activity_date,
    c.last_activity_date AS contact_last_activity_date,
    c.devzone_last_login_date,
    c.created_date AS contact_created_date,
    -- reference time for recency
    (SELECT ref_ts FROM params) AS ref_ts,
    -- raw aggregates
    COALESCE(f.n_activity_rows, 0) AS n_activity_rows,
    COALESCE(f.n_distinct_activity_id, 0) AS n_distinct_activity_id,
    f.first_activity_ts,
    f.last_activity_ts,
    COALESCE(f.n_active_days, 0) AS n_active_days,
    f.mean_score_capped,
    f.std_score_capped,
    f.max_score_capped,
    f.median_score_capped,
    COALESCE(f.n_distinct_activity_types, 0) AS n_distinct_activity_types,
    COALESCE(f.n_distinct_activity_roles, 0) AS n_distinct_activity_roles,
    COALESCE(f.n_high_score_events, 0) AS n_high_score_events,
    COALESCE(f.n_attendance_known, 0) AS n_attendance_known,
    COALESCE(f.n_attended_positive, 0) AS n_attended_positive,
    -- recency (days)
    CASE WHEN f.last_activity_ts IS NOT NULL
         THEN date_diff('day', CAST(f.last_activity_ts AS DATE), CAST((SELECT ref_ts FROM params) AS DATE))
    END AS days_since_last_activity,
    CASE WHEN c.devzone_last_login_date IS NOT NULL
         THEN date_diff('day', CAST(c.devzone_last_login_date AS DATE), CAST((SELECT ref_ts FROM params) AS DATE))
    END AS days_since_devzone_login,
    COALESCE(f.n_events_last_30d, 0) AS n_events_last_30d,
    COALESCE(f.n_events_last_90d, 0) AS n_events_last_90d,
    COALESCE(f.n_events_last_180d, 0) AS n_events_last_180d,
    COALESCE(f.n_events_last_365d, 0) AS n_events_last_365d,
    -- intensity
    CASE WHEN f.n_activity_rows > 0
         THEN CAST(f.n_activity_rows AS DOUBLE) / NULLIF(f.n_active_days, 0)
    END AS events_per_active_day,
    CASE
        WHEN f.first_activity_ts IS NOT NULL AND f.last_activity_ts IS NOT NULL
             AND f.last_activity_ts > f.first_activity_ts
        THEN GREATEST(
            1,
            date_diff('month', CAST(f.first_activity_ts AS DATE), CAST(f.last_activity_ts AS DATE)) + 1
        )
    END AS span_months_activity,
    CASE
        WHEN f.first_activity_ts IS NOT NULL AND f.last_activity_ts IS NOT NULL
             AND f.last_activity_ts > f.first_activity_ts
        THEN CAST(f.n_activity_rows AS DOUBLE)
             / NULLIF(GREATEST(1, date_diff('month', CAST(f.first_activity_ts AS DATE), CAST(f.last_activity_ts AS DATE)) + 1), 0)
    END AS events_per_span_month,
    CASE WHEN f.n_activity_rows > 0
         THEN CAST(f.n_events_last_90d AS DOUBLE) / f.n_activity_rows
    END AS share_events_last_90d,
    CASE WHEN f.n_activity_rows > 0
         THEN CAST(f.n_high_score_events AS DOUBLE) / f.n_activity_rows
    END AS share_high_score_events,
    f.sum_score_last_90d,
    CASE WHEN f.n_events_last_90d > 0
         THEN f.sum_score_last_90d / f.n_events_last_90d
    END AS mean_score_capped_last_90d,
    -- duration
    CASE
        WHEN f.first_activity_ts IS NOT NULL AND f.last_activity_ts IS NOT NULL
        THEN date_diff('day', CAST(f.first_activity_ts AS DATE), CAST(f.last_activity_ts AS DATE))
    END AS tenure_days_activity,
    CASE
        WHEN f.first_activity_ts IS NOT NULL AND f.last_activity_ts IS NOT NULL
             AND date_diff('day', CAST(f.first_activity_ts AS DATE), CAST(f.last_activity_ts AS DATE)) > 0
        THEN CAST(f.n_active_days AS DOUBLE) /
             (date_diff('day', CAST(f.first_activity_ts AS DATE), CAST(f.last_activity_ts AS DATE)) + 1)
    END AS active_day_density,
    -- quality / consistency
    CASE WHEN f.n_attendance_known > 0
         THEN CAST(f.n_attended_positive AS DOUBLE) / f.n_attendance_known
    END AS attendance_rate_where_known,
    CASE WHEN c.last_activity_date IS NOT NULL AND f.last_activity_ts IS NOT NULL
         THEN ABS(date_diff(
             'day',
             CAST(c.last_activity_date AS DATE),
             CAST(f.last_activity_ts AS DATE)
         ))
    END AS contact_vs_activity_last_date_gap_days
FROM contact_clean c
LEFT JOIN act_full f ON f.developer_id = c.developer_id;
"""

### Build table (writes to database)

Switch connection to read-write and run once; then set `read_only=True` again for exploration if you prefer.

In [ ]:
con.close()
con = duckdb.connect(DB_PATH, read_only=False)
con.execute(ENGAGEMENT_FEATURES_SQL)
print("developer_engagement_features created.")
con.execute("SELECT COUNT(*) AS n FROM developer_engagement_features").fetchdf()

### Preview and column list

In [ ]:
con.execute("DESCRIBE developer_engagement_features").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM developer_engagement_features
WHERE n_activity_rows > 100
ORDER BY n_events_last_90d DESC
LIMIT 8
""").fetchdf()

### Optional: export for modeling

```python
con.execute("COPY developer_engagement_features TO 'features/engagement.parquet' (FORMAT PARQUET)")
```